# Map/Reduce Evaluation with Confidence Metrics

Production pattern: compare documents individually (map), save results to JSONL, then aggregate into bulk metrics (reduce). Confidence metrics survive the round-trip.

In [ ]:
import json
import tempfile
from pathlib import Path
from typing import Optional

from stickler import (
    ComparableField,
    ExactComparator,
    LevenshteinComparator,
    NumericComparator,
    StructuredModel,
)
from stickler.structured_object_evaluator.bulk_structured_model_evaluator import (
    BulkStructuredModelEvaluator,
)

## 1. Define Model

In [ ]:
class Invoice(StructuredModel):
    invoice_number: str = ComparableField(comparator=ExactComparator(), threshold=1.0, weight=3.0)
    vendor: str = ComparableField(comparator=LevenshteinComparator(), threshold=0.7, weight=1.0)
    total: float = ComparableField(comparator=NumericComparator(tolerance=0.01), weight=2.0)
    notes: Optional[str] = ComparableField(comparator=LevenshteinComparator(), threshold=0.5, weight=0.5)

## 2. Simulate a Dataset

Ground truths are plain dicts. Predictions use the Rich Value Pattern with `_value` and `_confidence`.

In [ ]:
dataset = [
    {
        "doc_id": "INV-001",
        "gt": {"invoice_number": "INV-2024-001", "vendor": "Acme Corp", "total": 1247.50, "notes": "Delivered"},
        "pred": {
            "invoice_number": {"_value": "INV-2024-001", "_confidence": 0.97},
            "vendor": {"_value": "Acme Corporation", "_confidence": 0.72},
            "total": {"_value": 1247.50, "_confidence": 0.91},
            "notes": {"_value": "Left at door", "_confidence": 0.45},
        },
    },
    {
        "doc_id": "INV-002",
        "gt": {"invoice_number": "INV-2024-002", "vendor": "Globex Inc", "total": 530.00, "notes": "Signed"},
        "pred": {
            "invoice_number": {"_value": "INV-WRONG", "_confidence": 0.30},
            "vendor": {"_value": "Globex Inc", "_confidence": 0.88},
            "total": {"_value": 530.00, "_confidence": 0.95},
            "notes": {"_value": "Signed by manager", "_confidence": 0.60},
        },
    },
    {
        "doc_id": "INV-003",
        "gt": {"invoice_number": "INV-2024-003", "vendor": "Initech LLC", "total": 99.99},
        "pred": {
            "invoice_number": {"_value": "INV-2024-003", "_confidence": 0.99},
            "vendor": {"_value": "Initech", "_confidence": 0.65},
            "total": {"_value": 199.99, "_confidence": 0.25},
        },
    },
]

print(f"Dataset: {len(dataset)} documents")

## 3. Map Step: Compare Individual Documents, Save to JSONL

Each document is compared individually. The result (including `prediction_raw`) is saved to JSONL. This is the pattern used in distributed pipelines where each worker processes a subset of documents.

In [ ]:
# Use a temp file for this demo. NamedTemporaryFile avoids the toctou
# race that the deprecated tempfile.mktemp() is flagged for (bandit B306).
with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False) as _tf:
    jsonl_path = Path(_tf.name)

for doc in dataset:
    gt = Invoice(**doc["gt"])
    pred = Invoice.from_json(doc["pred"])  # from_json() preserves rich value metadata

    result = gt.compare_with(
        pred,
        include_confusion_matrix=True,
        document_field_comparisons=True,  # required for confidence metrics
    )

    # prediction_raw is automatically included when pred has rich values
    print(f"{doc['doc_id']}: score={result['overall_score']:.3f}, "
          f"has prediction_raw={'prediction_raw' in result}")

    # Save to JSONL
    with open(jsonl_path, "a") as f:
        record = {"doc_id": doc["doc_id"], "comparison_result": result}
        f.write(json.dumps(record, default=str) + "\n")

print(f"\nSaved {len(dataset)} results to {jsonl_path}")

## 4. Reduce Step: Aggregate from JSONL

Read the JSONL file and feed each result through `update_from_comparison_result()`. Confidence metrics are reconstructed from `prediction_raw` automatically.

In [ ]:
evaluator = BulkStructuredModelEvaluator(target_schema=Invoice)

with open(jsonl_path) as f:
    for line in f:
        record = json.loads(line)
        evaluator.update_from_comparison_result(
            record["comparison_result"],
            doc_id=record["doc_id"],
        )

result_from_jsonl = evaluator.compute()

print(f"Documents: {result_from_jsonl.document_count}")
print(f"Precision: {result_from_jsonl.metrics['cm_precision']:.3f}")
print(f"Recall:    {result_from_jsonl.metrics['cm_recall']:.3f}")
print(f"F1:        {result_from_jsonl.metrics['cm_f1']:.3f}")
print()
print(f"Confidence AUROC: {result_from_jsonl.confidence_metrics['overall']['auroc']['value']}")
print(f"Coverage: {result_from_jsonl.confidence_metrics['coverage']}")

## 5. Verify: Direct Bulk vs JSONL Replay

Both paths should produce identical metrics.

In [ ]:
# Direct bulk path (for comparison)
eval_direct = BulkStructuredModelEvaluator(target_schema=Invoice)
for doc in dataset:
    gt = Invoice(**doc["gt"])
    pred = Invoice.from_json(doc["pred"])
    eval_direct.update(gt, pred)

result_direct = eval_direct.compute()

# Compare
print(f"{'Metric':<25} {'Direct':>10} {'JSONL Replay':>12}")
print("-" * 50)
print(f"{'Precision':<25} {result_direct.metrics['cm_precision']:>10.3f} {result_from_jsonl.metrics['cm_precision']:>12.3f}")
print(f"{'Recall':<25} {result_direct.metrics['cm_recall']:>10.3f} {result_from_jsonl.metrics['cm_recall']:>12.3f}")
print(f"{'F1':<25} {result_direct.metrics['cm_f1']:>10.3f} {result_from_jsonl.metrics['cm_f1']:>12.3f}")

auroc_direct = result_direct.confidence_metrics['overall']['auroc']['value']
auroc_jsonl = result_from_jsonl.confidence_metrics['overall']['auroc']['value']
print(f"{'AUROC':<25} {auroc_direct:>10} {auroc_jsonl:>12}")

cov_direct = result_direct.confidence_metrics['coverage']
cov_jsonl = result_from_jsonl.confidence_metrics['coverage']
print(f"{'Coverage (with/total)':<25} {cov_direct['fields_with_confidence']}/{cov_direct['fields_total']:>5} {cov_jsonl['fields_with_confidence']}/{cov_jsonl['fields_total']:>8}")

print()
match = (result_direct.confidence_metrics['overall'] == result_from_jsonl.confidence_metrics['overall'])
print(f"Confidence metrics match: {match}")

## 6. Cleanup

In [ ]:
jsonl_path.unlink()
print("Temp file cleaned up.")

## Key Points

1. **Use `from_json()` for predictions** with rich values (`_value`, `_confidence`). This stores the raw JSON on the model.
2. **Call `compare_with()` with `document_field_comparisons=True`**. This ensures field-level comparison data is available.
3. **`prediction_raw` is included automatically** in the comparison result when the prediction has rich value metadata.
4. **`update_from_comparison_result()` reconstructs confidence pairs** from `prediction_raw`, producing identical metrics to the direct `update()` path.
5. **JSONL serialization preserves everything**. The map/reduce pattern works for confidence, and will work for future metadata types (bounding boxes, etc.).